# Lab 2 — SQLite MCP: Helios's Deal Tracker

Solstice's analysts log every diligence finding — a permit status, an interconnection cost
update, a covenant check — into a shared tracker as they work a deal. This notebook exposes that
tracker over MCP: once as **resources** (read this, don't call it like a function) and once as
**tools** (structured, parameterized queries a model can call).

## Concepts covered
- Exposing a SQLite database as an MCP **resource**, including a **parameterized resource
  template** (`tracker://table/{table}`) — these show up in a different listing call than static
  resources, which trips people up the first time.
- Exposing the same data as MCP **tools** for structured, parameterized queries.
- A **prompt** that turns a natural-language question into a query plan.
- A read-only guard so a "run any SQL" tool can't be used to mutate the database.
- Validating query results with plain `assert` checks (in place of a separate pytest file, so this
  notebook stays a single, runnable artifact).

In [1]:
# --- Single setup cell: dependencies, imports, and paths -------------------
%pip install --quiet --break-system-packages "mcp>=1.8.0" "python-dotenv>=1.0.1" "pydantic>=2.7.0"

import sqlite3
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
SERVERS_DIR = ROOT / "mcp_servers"
DATA_DIR.mkdir(parents=True, exist_ok=True)
SERVERS_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "helios_tracker.db"

print(f"ROOT        = {ROOT}")
print(f"DB_PATH     = {DB_PATH}")

Note: you may need to restart the kernel to use updated packages.
ROOT        = /home/claude/build/run/deliver/lab2
DB_PATH     = /home/claude/build/run/deliver/lab2/data/helios_tracker.db


## Step 1 — Build the tracker database

Three tables: `projects` (the acquisition targets), `diligence_findings` (one row per logged
finding, tagged with a category and a status), and `diligence_notes` (short internal notes on how
the tracker itself is used — status definitions, escalation rules, ownership). This is a plain
`sqlite3` script; nothing here is MCP-specific yet.

In [2]:
# --- Schema + seed data -----------------------------------------------------
SCHEMA_SQL = """
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS projects (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    technology TEXT NOT NULL,
    state TEXT NOT NULL,
    developer TEXT NOT NULL,
    capacity_mw REAL NOT NULL,
    stage TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS diligence_findings (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    project_id INTEGER NOT NULL,
    category TEXT NOT NULL,
    finding TEXT NOT NULL,
    status TEXT NOT NULL,
    logged_at TEXT DEFAULT (datetime('now')),
    FOREIGN KEY(project_id) REFERENCES projects(id)
);

CREATE TABLE IF NOT EXISTS diligence_notes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    topic TEXT NOT NULL,
    summary TEXT NOT NULL
);
"""

SEED_PROJECTS = [
    ("Cottonwood Solar", "solar", "TX", "Vantage Point Renewables", 150.0, "operating"),
    ("Prairie Horizon Wind", "wind", "KS", "Tall Grass Wind Co.", 210.0, "operating"),
    ("Sabine River Solar+Storage", "solar+storage", "TX", "Fernhollow Energy", 100.0, "late-stage development"),
    ("Redtail Ridge Wind", "wind", "OK", "Cascade Ridge Developers", 180.0, "construction"),
    ("Blue Mesa Solar", "solar", "NM", "Bluepeak Solar Partners", 120.0, "operating"),
]

# (project_name, category, finding, status) -- project_name is resolved to an id at insert time
SEED_FINDINGS = [
    ("Cottonwood Solar", "technical", "IE report confirms P50 capacity factor of 28.4%, in line with the underwriting model.", "clear"),
    ("Cottonwood Solar", "financial", "Q1 actual revenue trailing budget by 4% due to two unplanned inverter outages.", "monitor"),
    ("Cottonwood Solar", "environmental", "Updated stormwater permit renewed without conditions.", "clear"),
    ("Prairie Horizon Wind", "offtake", "12-year PPA with regional utility at $34/MWh; 3 years remain before merchant tail begins.", "monitor"),
    ("Prairie Horizon Wind", "environmental", "No open avian/bat mortality violations; annual monitoring report on file.", "clear"),
    ("Sabine River Solar+Storage", "interconnection", "Interconnection agreement fully executed; network upgrade cost finalized at $6.2M, in budget.", "clear"),
    ("Sabine River Solar+Storage", "legal", "Easement dispute with an adjacent landowner over the access road remains unresolved.", "flag"),
    ("Redtail Ridge Wind", "technical", "Turbine supply agreement delayed six weeks by a component shortage; revised COD Q1 2027.", "flag"),
    ("Redtail Ridge Wind", "financial", "Construction budget tracking 2% under approved contingency.", "clear"),
    ("Blue Mesa Solar", "offtake", "PPA expired last year; project now sells fully into the ERCOT merchant market.", "flag"),
    ("Blue Mesa Solar", "technical", "Degradation tracking slightly ahead of the IE's 0.5%/yr curve after year four.", "monitor"),
    ("Blue Mesa Solar", "financial", "Merchant exposure since PPA expiry has increased revenue volatility quarter over quarter.", "flag"),
]

SEED_NOTES = [
    ("Finding statuses", "clear = no follow-up needed; monitor = watch next quarter; flag = needs IC attention before close."),
    ("Diligence categories", "technical, financial, offtake, interconnection, environmental, legal -- every finding is tagged with exactly one."),
    ("Escalation rule", "Any flag open more than 14 days without a follow-up note is escalated to the deal lead automatically."),
    ("Data room structure", "Underlying PDFs (IE reports, permits, interconnection agreements) live in the project's data room folder; the tracker stores the findings extracted from them, not the documents themselves."),
    ("Refresh cadence", "Analysts update findings weekly during active diligence; Helios flags any project with no update in 10+ days as stale."),
    ("Ownership", "Technical and environmental findings are logged by the asset management team; financial and offtake by the investment team; legal by outside counsel via the deal lead."),
]


def initialize_tracker_db(db_path: Path = DB_PATH) -> Path:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        conn.executescript(SCHEMA_SQL)

        if conn.execute("SELECT COUNT(*) AS c FROM projects").fetchone()["c"] == 0:
            conn.executemany(
                "INSERT INTO projects(name, technology, state, developer, capacity_mw, stage) VALUES (?,?,?,?,?,?)",
                SEED_PROJECTS,
            )
        if conn.execute("SELECT COUNT(*) AS c FROM diligence_findings").fetchone()["c"] == 0:
            name_to_id = {row["name"]: row["id"] for row in conn.execute("SELECT id, name FROM projects")}
            conn.executemany(
                "INSERT INTO diligence_findings(project_id, category, finding, status) VALUES (?,?,?,?)",
                [(name_to_id[name], cat, finding, status) for name, cat, finding, status in SEED_FINDINGS],
            )
        if conn.execute("SELECT COUNT(*) AS c FROM diligence_notes").fetchone()["c"] == 0:
            conn.executemany("INSERT INTO diligence_notes(topic, summary) VALUES (?,?)", SEED_NOTES)

        conn.commit()
    return db_path


initialize_tracker_db()
print(f"Tracker DB ready at {DB_PATH}")

Tracker DB ready at /home/claude/build/run/deliver/lab2/data/helios_tracker.db


In [3]:
# --- Query helpers (the same logic the server will use) --------------------
from typing import Any


def list_tables(db_path: Path = DB_PATH) -> list[str]:
    with sqlite3.connect(db_path) as conn:
        rows = conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
        ).fetchall()
    return [r[0] for r in rows]


def describe_table(db_path: Path, table: str) -> list[dict]:
    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(f"PRAGMA table_info({table})").fetchall()
    return [dict(r) for r in rows]


def query_sqlite(db_path: Path, query: str, params: tuple = ()) -> list[dict]:
    """Only SELECT / WITH / PRAGMA are allowed -- this is the guard that keeps a
    'run any SQL' tool from being a 'drop any table' tool."""
    clean = query.strip().rstrip(";")
    if not clean.lower().startswith(("select", "with", "pragma")):
        raise ValueError("Only read-only queries are allowed through this tool.")
    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(clean, params).fetchall()
    return [dict(r) for r in rows]


def query_table(db_path: Path, table: str, limit: int = 10) -> list[dict]:
    if table not in {"projects", "diligence_findings", "diligence_notes"}:
        raise ValueError(f"Unknown table: {table}")
    return query_sqlite(db_path, f"SELECT * FROM {table} LIMIT ?", (limit,))


def database_overview(db_path: Path = DB_PATH) -> dict[str, Any]:
    return {"path": str(db_path), "tables": {t: describe_table(db_path, t) for t in list_tables(db_path)}}


print(list_tables())
print(query_table(DB_PATH, "projects", limit=2))

['diligence_findings', 'diligence_notes', 'projects']
[{'id': 1, 'name': 'Cottonwood Solar', 'technology': 'solar', 'state': 'TX', 'developer': 'Vantage Point Renewables', 'capacity_mw': 150.0, 'stage': 'operating'}, {'id': 2, 'name': 'Prairie Horizon Wind', 'technology': 'wind', 'state': 'KS', 'developer': 'Tall Grass Wind Co.', 'capacity_mw': 210.0, 'stage': 'operating'}]


## Step 2 — Expose it over MCP

`tracker_server.py` wraps the same database in a real MCP server:

- `tracker://overview` — a **static resource**: schema + column info for every table.
- `tracker://table/{table}` — a **parameterized resource template**. These are listed separately
  from static resources (`list_resource_templates()`, not `list_resources()`) — easy to miss the
  first time, so the client demo below calls both.
- `list_tables`, `query_table`, `run_select` — **tools** for structured and free-form (but
  read-only) queries.
- `sql_prompt` — a **prompt** that turns a natural-language question into a query plan.

In [4]:
%%writefile mcp_servers/tracker_server.py
"""Helios deal-tracker MCP server -- a standalone process wrapping data/helios_tracker.db."""
from __future__ import annotations

import sqlite3
from pathlib import Path
from typing import Any

from mcp.server.fastmcp import FastMCP

DB_PATH = Path("data/helios_tracker.db")
ALLOWED_TABLES = {"projects", "diligence_findings", "diligence_notes"}

mcp = FastMCP("helios-tracker")


def _list_tables() -> list[str]:
    with sqlite3.connect(DB_PATH) as conn:
        rows = conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
        ).fetchall()
    return [r[0] for r in rows]


def _describe_table(table: str) -> list[dict]:
    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(f"PRAGMA table_info({table})").fetchall()
    return [dict(r) for r in rows]


def _query(query: str, params: tuple = ()) -> list[dict]:
    clean = query.strip().rstrip(";")
    if not clean.lower().startswith(("select", "with", "pragma")):
        raise ValueError("Only read-only queries are allowed through this tool.")
    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(clean, params).fetchall()
    return [dict(r) for r in rows]


@mcp.resource("tracker://overview")
def tracker_overview() -> dict:
    """Schema + column info for every table in the tracker."""
    return {"path": str(DB_PATH), "tables": {t: _describe_table(t) for t in _list_tables()}}


@mcp.resource("tracker://table/{table}")
def tracker_table_resource(table: str) -> list[dict]:
    """Read up to 25 rows of a table as a resource (not a tool -- no side effects, just content)."""
    if table not in ALLOWED_TABLES:
        raise ValueError(f"Unknown table: {table}")
    return _query(f"SELECT * FROM {table} LIMIT 25")


@mcp.tool()
def list_tables() -> list[str]:
    """List tables available in the deal tracker."""
    return _list_tables()


@mcp.tool()
def query_table(table: str, limit: int = 10) -> list[dict]:
    """Query a specific tracker table safely, with a row limit."""
    if table not in ALLOWED_TABLES:
        raise ValueError(f"Unknown table: {table}")
    return _query(f"SELECT * FROM {table} LIMIT ?", (limit,))


@mcp.tool()
def run_select(query: str) -> list[dict]:
    """Run a read-only SQL query (SELECT / WITH / PRAGMA only) against the tracker."""
    return _query(query)


@mcp.prompt()
def sql_prompt(question: str, schema_hint: str = "") -> str:
    """Turn a natural-language tracker question into a structured query plan."""
    hint = f" Schema hint: {schema_hint}" if schema_hint else ""
    return (
        f"Answer this deal-tracker question step by step: {question}.{hint} "
        "Return: intent, tables involved, candidate SQL, and a one-line validation note."
    )


if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing mcp_servers/tracker_server.py


## Step 3 — Connect a real client

Static resources, resource *templates*, tools, and prompts are four separate listing calls. It's
easy to write a client that calls `list_resources()`, sees nothing relevant, and concludes the
server has no useful resources -- when the resource it wanted was actually a template.

In [5]:
# --- Raw MCP client session against the tracker server ---------------------
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

TRACKER_SERVER = StdioServerParameters(
    command="python3",
    args=[str(SERVERS_DIR / "tracker_server.py")],
    cwd=str(ROOT),
)


async def demo_tracker_session() -> None:
    async with stdio_client(TRACKER_SERVER) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            static_resources = await session.list_resources()
            print("static resources:", [str(r.uri) for r in static_resources.resources])

            templates = await session.list_resource_templates()
            print("resource templates:", [t.uriTemplate for t in templates.resourceTemplates])

            tools = await session.list_tools()
            print("tools:", [t.name for t in tools.tools])

            prompts = await session.list_prompts()
            print("prompts:", [p.name for p in prompts.prompts])

            print("\n--- read_resource('tracker://overview') tables ---")
            overview = await session.read_resource("tracker://overview")
            print(overview.contents[0].text[:300], "...")

            print("\n--- read_resource('tracker://table/projects') ---")
            table_res = await session.read_resource("tracker://table/projects")
            print(table_res.contents[0].text[:300], "...")

            print("\n--- call_tool('query_table', diligence_findings, limit=3) ---")
            findings = await session.call_tool("query_table", {"table": "diligence_findings", "limit": 3})
            # A tool that returns a bare list gets serialized as ONE CONTENT BLOCK PER ITEM,
            # unlike a dict return (which stays as a single block, as glossary_server's
            # search_glossary saw in Lab 1). Printing only content[0] here would silently show
            # just the first row and look like a limit=1 bug -- loop over every block instead.
            print(f"({len(findings.content)} content blocks for {len(findings.content)} rows)")
            for block in findings.content:
                print(block.text)

            print("\n--- get_prompt('sql_prompt') ---")
            prompt = await session.get_prompt(
                "sql_prompt", {"question": "Which projects have an open flag?", "schema_hint": "diligence_findings.status"}
            )
            print(prompt.messages[0].content.text)


await demo_tracker_session()

static resources: ['tracker://overview']
resource templates: ['tracker://table/{table}']
tools: ['list_tables', 'query_table', 'run_select']
prompts: ['sql_prompt']

--- read_resource('tracker://overview') tables ---
{
  "path": "data/helios_tracker.db",
  "tables": {
    "diligence_findings": [
      {
        "cid": 0,
        "name": "id",
        "type": "INTEGER",
        "notnull": 0,
        "dflt_value": null,
        "pk": 1
      },
      {
        "cid": 1,
        "name": "project_id",
        "type" ...

--- read_resource('tracker://table/projects') ---
[
  {
    "id": 1,
    "name": "Cottonwood Solar",
    "technology": "solar",
    "state": "TX",
    "developer": "Vantage Point Renewables",
    "capacity_mw": 150.0,
    "stage": "operating"
  },
  {
    "id": 2,
    "name": "Prairie Horizon Wind",
    "technology": "wind",
    "state": "KS",
     ...

--- call_tool('query_table', diligence_findings, limit=3) ---
(3 content blocks for 3 rows)
{
  "id": 1,
  "project_id": 

In [6]:
# --- Confirm the read-only guard actually holds ----------------------------
# A tool that runs "any SQL the model writes" is a tool that can also drop a table if a prompt
# injection or a bad model turn asks it to. Prove the guard rejects a write before trusting it.
async def try_write_query() -> None:
    async with stdio_client(TRACKER_SERVER) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("run_select", {"query": "DELETE FROM projects"})
            print("is_error:", result.isError)
            print(result.content[0].text)


await try_write_query()

is_error: True
Error executing tool run_select: Only read-only queries are allowed through this tool.


## Step 4 — Validate it

The original project workspace ran these checks with `pytest`. Since this notebook is meant to be
a single, self-contained artifact, the same checks run here as plain `assert` statements instead
-- same intent (catch a schema or seed-data regression before it reaches an analyst), no separate
test file to keep in sync.

In [7]:
# --- Inline validation (replaces a separate pytest file) -------------------
tables = list_tables(DB_PATH)
assert set(tables) == {"projects", "diligence_findings", "diligence_notes"}, f"unexpected tables: {tables}"

projects = query_sqlite(DB_PATH, "SELECT * FROM projects")
findings = query_sqlite(DB_PATH, "SELECT * FROM diligence_findings")
notes = query_sqlite(DB_PATH, "SELECT * FROM diligence_notes")

assert len(projects) == 5, f"expected 5 seed projects, got {len(projects)}"
assert len(findings) == 12, f"expected 12 seed findings, got {len(findings)}"
assert len(notes) == 6, f"expected 6 seed notes, got {len(notes)}"

project_ids = {p["id"] for p in projects}
orphaned = [f for f in findings if f["project_id"] not in project_ids]
assert not orphaned, f"found {len(orphaned)} findings pointing at a missing project"

allowed_statuses = {"clear", "monitor", "flag"}
bad_status = [f for f in findings if f["status"] not in allowed_statuses]
assert not bad_status, f"found findings with an unexpected status: {bad_status}"

sample_row = query_table(DB_PATH, "projects", limit=1)[0]
assert {"id", "name", "technology", "state", "developer", "capacity_mw", "stage"} <= sample_row.keys()

try:
    query_sqlite(DB_PATH, "DELETE FROM projects")
    raise AssertionError("expected the read-only guard to reject a DELETE")
except ValueError:
    pass  # expected

print(f"All checks passed: {len(projects)} projects, {len(findings)} findings, {len(notes)} notes.")

All checks passed: 5 projects, 12 findings, 6 notes.


## Verification

| Item | Status |
|---|---|
| `helios_tracker.db` schema + seed data | Sandbox-verified |
| `tracker_server.py` written and spawned as a subprocess | Sandbox-verified |
| Static resource (`tracker://overview`) | Sandbox-verified |
| Resource template (`tracker://table/{table}`) | Sandbox-verified, via `list_resource_templates()` |
| Tools (`list_tables`, `query_table`, `run_select`) | Sandbox-verified |
| Read-only guard against a write query | Sandbox-verified (rejected, `isError=True`) |
| `sql_prompt` prompt template | Sandbox-verified |
| Inline validation checks | Sandbox-verified, all assertions passed |
| MCP Inspector walkthrough | Manual step outside the notebook — needs Node.js |

## Edit points

- Add a table (e.g. `covenant_checks`) to `SCHEMA_SQL` and `SEED_*`, then add it to
  `ALLOWED_TABLES` in both this notebook and `mcp_servers/tracker_server.py`.
- Tighten or loosen the read-only guard in `_query` / `query_sqlite` if your own tools need a
  different safety boundary (e.g. allow `INSERT` into an `analyst_comments` table but nothing
  else).
- Add a second parameterized resource, like `tracker://project/{name}/findings`, to hand an agent
  a project-scoped view instead of the whole table.

**Next:** `lab3_multi_server_agent_solstice.ipynb` connects this tracker, a document-room server,
and PDF-backed retrieval into one LangGraph supervisor with a human approval gate.